# ResStock metadata and distribution plots

**Purpose:** metadata and distribution charts (heating fuel, HVAC type, cooling, building type, vintage, hot water) for the ResStock buildings matched to the AMI meters.

| # | Section | Key output |
|---|---------|-----------|
| 1 | Load county-filtered ResStock metadata | `df` |
| 2 | Metadata and distributions for matched buildings (2a-2d) | `results/plots/`, `results/plots_spot_ids/` |

Needs `results/match_report.csv` (AMI meter to ResStock IDs). Set `UPGRADE` in the "Upgrade selection" cell (0 = baseline; same value as in notebook 1). Run top to bottom.

## Setup
Assumes the project environment is already created and activated (`python -m pip install -r requirements.txt`). The two setup cells below are the same in every notebook of this project; run them first. Run notebooks from `grid_edge_fy25/notebooks/` (all paths are relative to it).

In [4]:
# Preflight check: assumes the environment was created from requirements.txt
import importlib.util

required_modules = ['pandas', 'numpy', 'matplotlib']
missing = [m for m in required_modules if importlib.util.find_spec(m) is None]
if missing:
    raise RuntimeError(
        "Missing Python packages: " + ", ".join(missing) +
        "\nCreate/activate the project environment and run: python -m pip install -r requirements.txt"
    )
print("Preflight OK: environment is ready.")


Preflight OK: environment is ready.


In [1]:
# Common setup (identical in all notebooks of this project)
import ast
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

# Paths (relative to grid_edge_fy25/notebooks/)
current_path = Path(os.getcwd())
data_path = current_path / ".." / "data"
base_files = data_path / "base_files"        # building id lists, measures, weather, workflow
input_files = data_path / "input_files"      # NY metadata, AMI consumption
parquet_folder = data_path / "parquet"       # load tables written for the optimization
results_path = current_path / ".." / "results"
extract_dir = data_path / "resstock_models"  # bldgXXXXXXX folders (models + simulation output)

parquet_folder.mkdir(parents=True, exist_ok=True)


---
## Upgrade selection
Set `UPGRADE` to the same value used in `1_resstock_models_download_and_run.ipynb` (0 = baseline). It selects the metadata file and tags the plot folders (`results/plots_up04/`, ...), so upgrades never overwrite the baseline plots. For an upgrade only buildings with `applicability == True` are used; matched buildings the upgrade does not apply to are dropped. Note the `in.*` columns plotted here describe the existing building, so they are the same for every upgrade; only the building set changes.

In [2]:
# ========================
# UPGRADE SELECTION (keep in sync with 1_resstock_models_download_and_run.ipynb)
# ========================
UPGRADE = 4          # 0 = baseline, 4 = ENERGY STAR heat pump with existing system as backup, ...

assert isinstance(UPGRADE, int) and UPGRADE >= 0, "UPGRADE must be a non-negative integer"
UPGRADE_TAG = "" if UPGRADE == 0 else f"_up{UPGRADE:02d}"   # baseline keeps the original names

METADATA_PATH = input_files / (
    "NY_baseline_metadata_and_annual_results_2024_amy.csv" if UPGRADE == 0
    else f"NY_upgrade{UPGRADE:02d}_metadata_and_annual_results_2024_amy.csv"
)
PLOTS_DIR      = results_path / f"plots{UPGRADE_TAG}"
PLOTS_SPOT_DIR = results_path / f"plots_spot_ids{UPGRADE_TAG}"

if not METADATA_PATH.exists():
    raise FileNotFoundError(f"Metadata for upgrade {UPGRADE} not found: {METADATA_PATH}")

print(f"UPGRADE = {UPGRADE} (tag '{UPGRADE_TAG}')")
print("Metadata:", METADATA_PATH.name)
print("Plots   :", PLOTS_DIR, "|", PLOTS_SPOT_DIR)


UPGRADE = 4 (tag '_up04')
Metadata: NY_upgrade04_metadata_and_annual_results_2024_amy.csv
Plots   : c:\Users\jxiong\Documents\GitHub\grid_edge_ithaca\grid_edge_fy25\notebooks\..\results\plots_up04 | c:\Users\jxiong\Documents\GitHub\grid_edge_ithaca\grid_edge_fy25\notebooks\..\results\plots_spot_ids_up04


---
## 1. Load ResStock metadata
Reads the NY ResStock metadata for the selected `UPGRADE` and keeps Cayuga, Chemung, Cortland, Schuyler, Seneca, Tioga and Tompkins counties (for upgrades, only applicable buildings). Defines `df`, used by section 2.

In [3]:
# County-filtered ResStock metadata for the selected UPGRADE (same filter as section 1 of 1_resstock_models_download_and_run.ipynb); defines `df`
buildstock_metadata_path = METADATA_PATH

selected_counties = [
    "Cayuga County",
    "Chemung County",
    "Cortland County",
    "Schuyler County",
    "Seneca County",
    "Tioga County",
    "Tompkins County",
]

df = pd.read_csv(buildstock_metadata_path, low_memory=False)
df = df[df["in.county_name"].isin(selected_counties)]
if UPGRADE != 0:   # upgrade models exist only for buildings the upgrade applies to
    df = df[df["applicability"].astype(bool)]
print(f"Upgrade {UPGRADE}: metadata rows:", len(df))
df


Upgrade 4: metadata rows: 483


,bldg_id,upgrade,weight,upgrade_name,applicability,in.sqft,in.representative_income,in.ahs_region,in.aiannh_area,in.area_median_income,...,upgrade.cooling_setpoint_offset_magnitude,upgrade.cooling_setpoint_offset_period,upgrade.heating_setpoint_has_offset,upgrade.heating_setpoint_offset_magnitude,upgrade.heating_setpoint_offset_period,upgrade.hvac_cooling_efficiency,upgrade.hvac_cooling_partial_space_conditioning,upgrade.hvac_heating_efficiency,upgrade.hvac_secondary_heating_efficiency,upgrade.hvac_secondary_heating_fuel
33,510,4,252.301639,ENERGY STAR heat pump with existing system as ...,True,2179,47457.0,Non-CBSA Middle Atlantic,No,60-80%,...,NaN,NaN,NaN,NaN,NaN,Ducted Heat Pump,100% Conditioned,"Dual-Fuel ASHP, SEER 16, 9.2 HSPF, Integrated ...",NaN,NaN
110,1700,4,252.301639,ENERGY STAR heat pump with existing system as ...,True,3310,109096.0,Non-CBSA Middle Atlantic,No,150%+,...,NaN,NaN,NaN,NaN,NaN,Ducted Heat Pump,100% Conditioned,"Dual-Fuel ASHP, SEER 16, 9.2 HSPF, Integrated ...",NaN,NaN
210,3424,4,252.301639,ENERGY STAR heat pump with existing system as ...,True,3310,280058.0,Non-CBSA Middle Atlantic,No,150%+,...,NaN,NaN,NaN,NaN,NaN,Ducted Heat Pump,100% Conditioned,"Dual-Fuel ASHP, SEER 16, 9.2 HSPF, Integrated ...",NaN,NaN
382,6034,4,252.301639,ENERGY STAR heat pump with existing system as ...,True,1698,0.0,Non-CBSA Middle Atlantic,No,Not Available,...,NaN,NaN,NaN,NaN,NaN,Ducted Heat Pump,100% Conditioned,"Dual-Fuel ASHP, SEER 16, 9.2 HSPF, Integrated ...",NaN,NaN
399,6232,4,252.301639,ENERGY STAR heat pump with existing system as ...,True,2179,270137.0,Non-CBSA Middle Atlantic,No,150%+,...,NaN,NaN,NaN,NaN,NaN,Ducted Heat Pump,100% Conditioned,"Dual-Fuel ASHP, SEER 16, 9.2 HSPF, Integrated ...",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33426,543959,4,252.301639,ENERGY STAR heat pump with existing system as ...,True,2678,273335.0,Non-CBSA Middle Atlantic,No,150%+,...,NaN,NaN,NaN,NaN,NaN,Ducted Heat Pump,100% Conditioned,"Dual-Fuel ASHP, SEER 16, 9.2 HSPF, Integrated ...",NaN,NaN
33605,547188,4,252.301639,ENERGY STAR heat pump with existing system as ...,True,1698,42401.0,Non-CBSA Middle Atlantic,No,30-60%,...,0F,NaN,No,0F,NaN,Non-Ducted Heat Pump,100% Conditioned,"MSHP, SEER 16, 9.2 HSPF, Separate Backup, Max ...","Fuel Wall/Floor Furnace, 68% AFUE",Natural Gas
33620,547386,4,252.301639,ENERGY STAR heat pump with existing system as ...,True,1698,89096.0,Non-CBSA Middle Atlantic,No,120-150%,...,NaN,NaN,NaN,NaN,NaN,Ducted Heat Pump,100% Conditioned,"Dual-Fuel ASHP, SEER 16, 9.2 HSPF, Integrated ...",NaN,NaN
33621,547426,4,252.301639,ENERGY STAR heat pump with existing system as ...,True,1228,22473.0,Non-CBSA Middle Atlantic,No,30-60%,...,NaN,NaN,NaN,NaN,NaN,Ducted Heat Pump,100% Conditioned,"Dual-Fuel ASHP, SEER 16, 9.2 HSPF, Integrated ...",NaN,NaN


---
## 2. Metadata and distributions for matched ResStock buildings
**2a.** Expand `match_report.csv` (AMI meter to ResStock IDs, duplicates kept as weights) and merge with metadata into `df_metadata_weighted`.

In [ ]:
## CODE BLOCK AGGREGATE BUILDING METADATA WITH SELECTED RESSOTCK IDS

# Path to matched ReStock IDs
# TODO : Update path
match_report_path = current_path / '..'/ 'results' / 'match_report.csv'

# Read the matched resstock ID names
df_match = pd.read_csv(match_report_path)

# --- 2. Parse ResStock_ids column ---
def parse_ids(x):
    if pd.isna(x): return []
    try:
        arr = ast.literal_eval(str(x)) if isinstance(x, str) else []
        return arr if isinstance(arr, list) else []
    except Exception:
        return []

# Flatten all ResStock IDs (keeping duplicates)
resstock_ids_all = (
    df_match['ResStock_ids']
    .dropna()
    .apply(parse_ids)
    .explode()
    .dropna()
    .astype(int)
)

print(f"Total ResStock IDs (including duplicates): {len(resstock_ids_all)}")

# --- 3. Directly merge each ID with its metadata ---
# Turn the ResStock IDs (with duplicates) into a DataFrame
df_resstock = pd.DataFrame({'bldg_id': resstock_ids_all})

# Merge — this automatically duplicates metadata for repeated IDs
df_metadata_weighted = df_resstock.merge(df, on='bldg_id', how='left')
if UPGRADE != 0:   # matched buildings the upgrade does not apply to have no metadata row: drop them
    df_metadata_weighted = df_metadata_weighted.dropna(subset=['in.county_name'])

# --- 4. Quick sanity checks ---
print(f"Final df_metadata_weighted rows: {len(df_metadata_weighted)}")
print(f"Unique bldg_ids: {df_metadata_weighted['bldg_id'].nunique()}")


df_metadata_weighted

**2b.** Distribution charts (heating fuel, HVAC type, cooling, building type, vintage, hot water) for all matched buildings. Saved to `results/plots/`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Folder to save charts
#TODO: Update Path
output_dir = PLOTS_DIR   # results/plots[_upNN], from the 'Upgrade selection' cell
output_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# FIXED COLOR MAPS (for pie charts)
# -----------------------------
heating_fuel_colors = {
    "Natural Gas": "#1f77b4",   # blue
    "Electricity": "#ff7f0e",   # orange
    "Propane": "#2ca02c",       # green
    "Fuel Oil": "#d62728",      # red
    "Other Fuel": "#9467bd"     # purple
}

cooling_type_colors = {
    "Room AC": "#1f77b4",               # blue
    "Central AC": "#ff7f0e",            # orange
    "Ducted Heat Pump": "#2ca02c",      # green
    "Non-Ducted Heat Pump": "#d62728"   # red
}

water_heater_fuel_colors = {
    "Natural Gas": "#1f77b4",
    "Electricity": "#ff7f0e",
    "Propane": "#2ca02c",
    "Fuel Oil": "#d62728",
    "Other Fuel": "#9467bd"
}

building_type_colors = {
    "Single-Family Detached": "#1f77b4",
    "Single-Family Attached": "#ff7f0e",
    "Multi-Family with 2 - 4 Units": "#2ca02c",
    "Multi-Family with 5+ Units": "#d62728",
    "Mobile Home": "#9467bd"
}

vintage_colors = {
    "<1940": "#1f77b4",
    "1940s": "#ff7f0e",
    "1950s": "#2ca02c",
    "1960s": "#d62728",
    "1970s": "#9467bd",
    "1980s": "#8c564b",
    "1990s": "#e377c2",
    "2000s": "#7f7f7f",
    "2010s": "#bcbd22"
}

# -----------------------------
# Fuel-family color scheme for horizontal bar charts
# -----------------------------
FUEL_BASE = {
    "Natural Gas": "#1f77b4",
    "Electricity": "#ff7f0e",
    "Electric":    "#ff7f0e",   # water heater categories say "Electric ..."
    "Propane":     "#2ca02c",
    "Fuel Oil":    "#d62728",
    "Other Fuel":  "#9467bd",
    "Other":       "#9467bd",
    "Wood":        "#8c564b",
    "Solar":       "#bcbd22",
    "None":        "#cccccc",
}

def fuel_of(category: str) -> str:
    """Map a long category string to its fuel family."""
    c = category.lower()
    if c.startswith("natural gas"): return "Natural Gas"
    if c.startswith("electric"):    return "Electricity"
    if c.startswith("propane"):     return "Propane"
    if "fuel oil" in c:             return "Fuel Oil"
    if c.startswith("solar"):       return "Solar"
    if c.startswith("wood"):        return "Wood"
    if c.startswith("other"):       return "Other Fuel"
    if c == "none":                 return "None"
    return "Other Fuel"

def _shade(hex_color: str, factor: float) -> str:
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    if factor >= 1:
        r = int(r + (255 - r) * (factor - 1))
        g = int(g + (255 - g) * (factor - 1))
        b = int(b + (255 - b) * (factor - 1))
    else:
        r, g, b = int(r * factor), int(g * factor), int(b * factor)
    return f"#{max(0,min(255,r)):02x}{max(0,min(255,g)):02x}{max(0,min(255,b)):02x}"

def auto_color_map(categories):
    """Auto-assign shades within each fuel family."""
    groups = {}
    for cat in categories:
        groups.setdefault(fuel_of(cat), []).append(cat)
    cmap = {}
    for fuel, cats in groups.items():
        base = FUEL_BASE.get(fuel, "#888888")
        n = len(cats)
        factors = np.linspace(0.75, 1.3, n) if n > 1 else [1.0]
        for cat, f in zip(cats, factors):
            cmap[cat] = _shade(base, f)
    return cmap


# -----------------------------
# Helpers
# -----------------------------
def get_colors(data_index, color_map, default_color="#cccccc"):
    return [color_map.get(cat, default_color) for cat in data_index]


def plot_and_save_pie(data, title, filename, color_map=None):
    """Create, show, and save a clean pie chart with consistent colors."""
    if not data.empty:
        plt.figure(figsize=(8, 8))
        data_rounded = data.round(0)
        colors = get_colors(data_rounded.index, color_map) if color_map else None

        wedges, texts, autotexts = plt.pie(
            data_rounded,
            colors=colors,
            autopct=lambda p: f'{int(round(p))}%' if p >= 2 else '',
            startangle=90,
            labeldistance=1.2,
            pctdistance=0.75,
            wedgeprops={'linewidth': 1, 'edgecolor': 'white'}
        )
        plt.legend(
            wedges,
            data_rounded.index,
            title="Categories",
            loc="center left",
            bbox_to_anchor=(1, 0, 0.5, 1),
            fontsize=12
        )
        plt.setp(autotexts, size=12, weight="bold", color="white")
        plt.title(title, fontsize=16, weight='bold')
        plt.tight_layout()

        save_path = output_dir / filename
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved: {save_path}")
        plt.show()


def plot_h_bar_sorted(series, title, filename):
    """Horizontal bar chart sorted by share, colored by fuel family."""
    if series.empty:
        return
    s = series.sort_values(ascending=True)
    cmap = auto_color_map(s.index)
    colors = [cmap[c] for c in s.index]

    fig, ax = plt.subplots(figsize=(11, max(5, 0.38 * len(s))))
    bars = ax.barh(s.index, s.values, color=colors, edgecolor="white")
    ax.set_xlabel("Share (%)", fontsize=12)
    ax.set_title(title, fontsize=15, weight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlim(0, max(s.values) * 1.12)

    for bar, v in zip(bars, s.values):
        ax.text(v + max(s.values) * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f"{v:.1f}%", va="center", fontsize=10)

    plt.tight_layout()
    save_path = output_dir / filename
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"Saved: {save_path}")
    plt.show()


# --- Heating Fuel Pie Chart ---
if 'in.heating_fuel' in df_metadata_weighted.columns:
    heating_dist = (
        df_metadata_weighted['in.heating_fuel']
        .value_counts(normalize=True) * 100
    )
    plot_and_save_pie(
        heating_dist,
        "Heating Fuel Distribution",
        "heating_fuel_distribution.png",
        color_map=heating_fuel_colors
    )

# # --- HVAC Heating Type + Fuel: table + horizontal bar chart ---
# if 'in.hvac_heating_type_and_fuel' in df_metadata_weighted.columns:
#     dist_table = (
#         df_metadata_weighted['in.hvac_heating_type_and_fuel']
#         .dropna()
#         .value_counts()
#         .rename_axis('hvac_heating_type_and_fuel')
#         .reset_index(name='count')
#     )
#     dist_table['percent'] = (
#         dist_table['count'] / dist_table['count'].sum() * 100
#     ).round(1)
#     print("\nHVAC Heating Type + Fuel Distribution:")
#     print(dist_table.to_string(index=False))
#     csv_path = output_dir / "hvac_heating_type_and_fuel_distribution.csv"
#     dist_table.to_csv(csv_path, index=False)
#     print(f"Saved table: {csv_path}")

#     hvac_heating_dist = (
#         df_metadata_weighted['in.hvac_heating_type_and_fuel']
#         .dropna()
#         .value_counts(normalize=True) * 100
#     )
#     plot_h_bar_sorted(
#         hvac_heating_dist,
#         "HVAC Heating Type and Fuel Distribution",
#         "hvac_heating_type_and_fuel_distribution.png"
#     )
# --- GrIDS HVAC Type: table + horizontal bar chart ---
if "GrIDS HVAC Type" in df_metadata_weighted.columns:
    grids_hvac_table = (
        df_metadata_weighted["GrIDS HVAC Type"]
        .dropna()
        .value_counts()
        .rename_axis("grids_hvac_type")
        .reset_index(name="count")
    )
    grids_hvac_table["percent"] = (
        grids_hvac_table["count"] / grids_hvac_table["count"].sum() * 100
    ).round(1)

    print("\nGrIDS HVAC Type Distribution:")
    print(grids_hvac_table.to_string(index=False))

    csv_path = output_dir / "grids_hvac_type_distribution.csv"
    grids_hvac_table.to_csv(csv_path, index=False)
    print(f"Saved table: {csv_path}")

    grids_hvac_dist = (
        df_metadata_weighted["GrIDS HVAC Type"]
        .dropna()
        .value_counts(normalize=True) * 100
    )

    plot_h_bar_sorted(
        grids_hvac_dist,
        "GrIDS HVAC Type Distribution",
        "grids_hvac_type_distribution.png"
    )
# --- Cooling Type Pie Chart ---
if 'in.hvac_cooling_type' in df_metadata_weighted.columns:
    cooling_dist = (
        df_metadata_weighted['in.hvac_cooling_type']
        .value_counts(normalize=True) * 100
    )
    plot_and_save_pie(
        cooling_dist,
        "Cooling Type Distribution",
        "cooling_type_distribution.png",
        color_map=cooling_type_colors
    )

# --- Building Type (RECS) Pie Chart ---
if 'in.geometry_building_type_recs' in df_metadata_weighted.columns:
    building_type_dist = (
        df_metadata_weighted['in.geometry_building_type_recs']
        .value_counts(normalize=True) * 100
    )
    plot_and_save_pie(
        building_type_dist,
        "Building Type (RECS) Distribution",
        "building_type_recs_distribution.png",
        color_map=building_type_colors
    )

# --- Building Vintage Pie Chart ---
if 'in.vintage' in df_metadata_weighted.columns:
    vintage_dist = (
        df_metadata_weighted['in.vintage']
        .value_counts(normalize=True) * 100
    )
    plot_and_save_pie(
        vintage_dist,
        "Building Vintage Distribution",
        "building_vintage_distribution.png",
        color_map=vintage_colors
    )

# --- Water Heater Fuel Pie Chart ---
if 'in.water_heater_fuel' in df_metadata_weighted.columns:
    water_heater_dist = (
        df_metadata_weighted['in.water_heater_fuel']
        .value_counts(normalize=True) * 100
    )
    plot_and_save_pie(
        water_heater_dist,
        "Domestic Hot Water Fuel Distribution",
        "domestic_hot_water_distribution.png",
        color_map=water_heater_fuel_colors
    )

# # --- Water Heater Efficiency: table + horizontal bar chart ---
# if 'in.water_heater_efficiency' in df_metadata_weighted.columns:
#     wh_eff_table = (
#         df_metadata_weighted['in.water_heater_efficiency']
#         .dropna()
#         .value_counts()
#         .rename_axis('water_heater_efficiency')
#         .reset_index(name='count')
#     )
#     wh_eff_table['percent'] = (
#         wh_eff_table['count'] / wh_eff_table['count'].sum() * 100
#     ).round(1)
#     print("\nWater Heater Efficiency Distribution:")
#     print(wh_eff_table.to_string(index=False))
#     csv_path = output_dir / "water_heater_efficiency_distribution.csv"
#     wh_eff_table.to_csv(csv_path, index=False)
#     print(f"Saved table: {csv_path}")

#     wh_eff_dist = (
#         df_metadata_weighted['in.water_heater_efficiency']
#         .dropna()
#         .value_counts(normalize=True) * 100
#     )
#     plot_h_bar_sorted(
#         wh_eff_dist,
#         "Water Heater Efficiency Distribution",
#         "water_heater_efficiency_distribution.png"
    # )

    # --- GrIDS Hot Water Type: table + horizontal bar chart ---
if "GrIDS hot Water Type" in df_metadata_weighted.columns:
    grids_hw_table = (
        df_metadata_weighted["GrIDS hot Water Type"]
        .dropna()
        .value_counts()
        .rename_axis("grids_hot_water_type")
        .reset_index(name="count")
    )
    grids_hw_table["percent"] = (
        grids_hw_table["count"] / grids_hw_table["count"].sum() * 100
    ).round(1)

    print("\nGrIDS Hot Water Type Distribution:")
    print(grids_hw_table.to_string(index=False))

    csv_path = output_dir / "grids_hot_water_type_distribution.csv"
    grids_hw_table.to_csv(csv_path, index=False)
    print(f"Saved table: {csv_path}")

    grids_hw_dist = (
        df_metadata_weighted["GrIDS hot Water Type"]
        .dropna()
        .value_counts(normalize=True) * 100
    )

    plot_h_bar_sorted(
        grids_hw_dist,
        "GrIDS Hot Water Type Distribution",
        "grids_hot_water_type_distribution.png"
    )

**2c.** Same distributions, restricted to a chosen set of AMI spot (meter) IDs (e.g. one feeder branch). Edit `spot_ids` in the cell. Outputs go to `results/plots_spot_ids/`.

In [ ]:
import os
import ast
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# =========================
# INPUT PATHS
# =========================
current_path = Path(os.getcwd())
data_path = current_path / ".." / "data"

buildstock_metadata_path = METADATA_PATH   # from the 'Upgrade selection' cell
match_report_path = current_path / ".." / "results" / "match_report.csv"

# Output folder
output_dir = PLOTS_SPOT_DIR   # results/plots_spot_ids[_upNN], from the 'Upgrade selection' cell
output_dir.mkdir(parents=True, exist_ok=True)

# =========================
# FIXED COLOR MAPS
# =========================
heating_fuel_colors = {
    "Natural Gas": "#1f77b4",   # blue
    "Electricity": "#ff7f0e",   # orange
    "Fuel Oil": "#2ca02c",      # green
    "Propane": "#d62728",       # red
    "Other Fuel": "#9467bd"     # purple
}

cooling_type_colors = {
    "Room AC": "#1f77b4",               # blue
    "Central AC": "#ff7f0e",            # orange
    "Ducted Heat Pump": "#2ca02c",      # green
    "Non-Ducted Heat Pump": "#d62728"   # red
}

water_heater_fuel_colors = {
    "Natural Gas": "#1f77b4",   # blue
    "Electricity": "#ff7f0e",   # orange
    "Fuel Oil": "#2ca02c",      # green
    "Propane": "#d62728",       # red
    "Other Fuel": "#9467bd"     # purple
}

# =========================
# SPOT IDS OF INTEREST
# =========================
# BRANCH 4
spot_ids = [
    6000973888,
    6000973966,
    6000973949,
    6001002030,
    6000973941,
    6000973894,
    6003326927,
    6000973865,
    6003553270,
    6000974002
]

# # BRANCH 3
# spot_ids = [
#     6003454683,
#     6000990407,
#     6000990458,
#     6000990470,
#     6000990441,
#     6000990345,
#     6000990399,
#     6000990384,
#     6000990369,
#     6000998866
# ]

# =========================
# READ DATA
# =========================
# df_metadata = pd.read_excel(buildstock_metadata_path, sheet_name="select_counties")
df_metadata = df
df_match = pd.read_csv(match_report_path)

print("Metadata rows:", len(df_metadata))
print("Match report rows:", len(df_match))

print("\nMatch report columns:")
print(df_match.columns.tolist())

# =========================
# IDENTIFY SPOT/METER COLUMN
# =========================
possible_meter_cols = ["meter id", "#meter id", "meter_id", "spot_id", "Spot Number"]

meter_col = None
for col in possible_meter_cols:
    if col in df_match.columns:
        meter_col = col
        break

if meter_col is None:
    raise ValueError(
        f"Could not find meter/spot column in match report. "
        f"Available columns: {df_match.columns.tolist()}"
    )

print(f"\nUsing meter column: {meter_col}")

df_match[meter_col] = pd.to_numeric(df_match[meter_col], errors="coerce")

# =========================
# FILTER TO REQUIRED SPOT IDS
# =========================
df_match_spots = df_match[df_match[meter_col].isin(spot_ids)].copy()

print(f"\nMatched rows for selected spot IDs: {len(df_match_spots)}")
print("Spot IDs found in match report:", sorted(df_match_spots[meter_col].dropna().unique().tolist()))

missing_spots = sorted(set(spot_ids) - set(df_match_spots[meter_col].dropna().astype(int).tolist()))
if missing_spots:
    print("Spot IDs missing from match report:", missing_spots)

df_match_spots.to_csv(output_dir / "filtered_match_report_selected_spots.csv", index=False)

# =========================
# PARSE ResStock_ids
# =========================
def parse_ids(x):
    if pd.isna(x):
        return []
    try:
        arr = ast.literal_eval(str(x)) if isinstance(x, str) else []
        return arr if isinstance(arr, list) else []
    except Exception:
        return []

df_resstock_expanded = df_match_spots[[meter_col, "ResStock_ids"]].copy()
df_resstock_expanded["ResStock_ids"] = df_resstock_expanded["ResStock_ids"].apply(parse_ids)
df_resstock_expanded = df_resstock_expanded.explode("ResStock_ids").dropna(subset=["ResStock_ids"]).copy()
df_resstock_expanded["bldg_id"] = pd.to_numeric(df_resstock_expanded["ResStock_ids"], errors="coerce").astype("Int64")
df_resstock_expanded = df_resstock_expanded.dropna(subset=["bldg_id"]).copy()
df_resstock_expanded["bldg_id"] = df_resstock_expanded["bldg_id"].astype(int)

print(f"\nTotal matched ResStock IDs for selected spots (including duplicates): {len(df_resstock_expanded)}")
print(f"Unique matched ResStock IDs: {df_resstock_expanded['bldg_id'].nunique()}")

df_resstock_expanded.to_csv(output_dir / "selected_spots_resstock_ids_expanded.csv", index=False)

# =========================
# MERGE WITH METADATA
# =========================
df_metadata_weighted = df_resstock_expanded.merge(df_metadata, on="bldg_id", how="left")
if UPGRADE != 0:   # matched buildings the upgrade does not apply to have no metadata row: drop them
    df_metadata_weighted = df_metadata_weighted.dropna(subset=["in.county_name"])

print(f"\nMerged metadata rows: {len(df_metadata_weighted)}")
print(f"Unique bldg_ids after merge: {df_metadata_weighted['bldg_id'].nunique()}")

missing_metadata = df_metadata_weighted["bldg_id"].isna().sum()
print(f"Rows with missing metadata after merge: {missing_metadata}")

df_metadata_weighted.to_csv(output_dir / "selected_spots_metadata_weighted.csv", index=False)

# =========================
# OPTIONAL: PER-SPOT COUNTS
# =========================
spot_summary = (
    df_resstock_expanded.groupby(meter_col)["bldg_id"]
    .agg(["count", "nunique"])
    .reset_index()
    .rename(columns={"count": "total_matches", "nunique": "unique_resstock_ids"})
)

print("\nPer-spot summary:")
print(spot_summary.to_string(index=False))

spot_summary.to_csv(output_dir / "selected_spots_summary.csv", index=False)

# =========================
# PLOTTING HELPERS
# =========================
def get_colors(labels, col_name):
    if col_name == "in.heating_fuel":
        cmap = heating_fuel_colors
    elif col_name == "in.hvac_cooling_type":
        cmap = cooling_type_colors
    elif col_name == "in.water_heater_fuel":
        cmap = water_heater_fuel_colors
    else:
        cmap = {}

    return [cmap.get(label, "#cccccc") for label in labels]


def plot_and_save_pie(data, title, filename, legend_title="Categories", col_name=None):
    if data.empty:
        print(f"Skipped {title} because data is empty.")
        return

    plt.figure(figsize=(8, 8))

    data_rounded = data.round(0)
    colors = get_colors(data_rounded.index, col_name)

    wedges, texts, autotexts = plt.pie(
        data_rounded,
        colors=colors,
        autopct=lambda p: f"{int(round(p))}%" if p >= 2 else "",
        startangle=90,
        pctdistance=0.75,
        wedgeprops={"linewidth": 1, "edgecolor": "white"}
    )

    plt.legend(
        wedges,
        data_rounded.index,
        title=legend_title,
        loc="center left",
        bbox_to_anchor=(1, 0, 0.5, 1),
        fontsize=10
    )

    plt.setp(autotexts, size=11, weight="bold", color="white")
    plt.title(title, fontsize=15, weight="bold")
    plt.tight_layout()

    save_path = output_dir / filename
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"Saved: {save_path}")
    plt.close()

# =========================
# DISTRIBUTIONS + PLOTS
# =========================
plot_columns = {
    "in.heating_fuel": ("Heating Fuel Distribution", "heating_fuel_distribution.png"),
    "in.hvac_heating_type_and_fuel": ("HVAC Heating Type + Fuel Distribution", "hvac_heating_type_and_fuel_distribution.png"),
    "in.hvac_cooling_type": ("Cooling Type Distribution", "cooling_type_distribution.png"),
    "in.geometry_building_type_recs": ("Building Type (RECS) Distribution", "building_type_recs_distribution.png"),
    "in.vintage": ("Building Vintage Distribution", "building_vintage_distribution.png"),
    "in.water_heater_fuel": ("Domestic Hot Water Fuel Distribution", "domestic_hot_water_distribution.png"),
}

for col, (title, filename) in plot_columns.items():
    if col in df_metadata_weighted.columns:
        dist = df_metadata_weighted[col].dropna().value_counts(normalize=True) * 100

        dist_table = (
            df_metadata_weighted[col]
            .dropna()
            .value_counts()
            .rename_axis(col)
            .reset_index(name="count")
        )
        dist_table["percent"] = (dist_table["count"] / dist_table["count"].sum() * 100).round(1)

        csv_name = filename.replace(".png", ".csv")
        dist_table.to_csv(output_dir / csv_name, index=False)
        print(f"Saved table: {output_dir / csv_name}")

        plot_and_save_pie(
            dist,
            title,
            filename,
            legend_title=col,
            col_name=col
        )
    else:
        print(f"Column not found, skipped: {col}")

print("\nDone.")

**2d.** Quick sanity check of the `meter id` column in the match report (optional).

In [ ]:
import pandas as pd

df_match = pd.read_csv(match_report_path)

print(df_match['meter id'].dtype)
print(df_match['meter id'].head(20).tolist())
print(df_match['meter id'].tail(20).tolist())
print(df_match['meter id'].nunique())